# Brescia Transit Network - Food Delivery Optimization
This notebook models the Brescia metro and tram network, generates restaurant/customer locations, and solves an order assignment optimization problem using integer programming.

## 1. Imports & Setup

In [ ]:
from datetime import datetime, time
import heapq
import numpy as np
import random
import osmnx as ox
import matplotlib.pyplot as plt
from geopy.distance import geodesic
import pyomo.environ as pyo
from pyomo.environ import *
from pyomo.environ import ConstraintList
from pyomo.opt import SolverFactory

## 2. Network Definition: Stations & Arcs

In [ ]:
# Bounding box for Brescia area
north, south, east, west = 45.59000, 45.51000, 10.28100, 10.17000
bbox = (north, south, east, west)

# Metro Stations
metro_stations = {
    'Prealpino':              (45.58099, 10.22928),
    'Casazza':                (45.57598, 10.23030),
    'Mompiano':               (45.56894, 10.23669),
    'Europa':                 (45.56413, 10.23429),
    'Ospedale':               (45.55646, 10.23349),
    'Marconi':                (45.55231, 10.22468),
    'San Faustino':           (45.54567, 10.22000),
    'Vittoria':               (45.53942, 10.21847),
    'Stazione FS':            (45.53334, 10.21455),
    'Bresciadue':             (45.52736, 10.21280),
    'A2A Lamarmora':          (45.51906, 10.22550),
    'Volta':                  (45.51882, 10.22542),
    'Poliambulanza':          (45.51859, 10.23673),
    'San Polo Parco':         (45.51778, 10.24255),
    'San Polo':               (45.51966, 10.25910),
    'Sanpolino':              (45.51746, 10.26689),
    "Sant'Eufemia Buffalora": (45.51145, 10.28007),
}

# Tram Stations
tram_stations = {
    'Tram 1':  (45.569,    10.20707),
    'Tram 2':  (45.56498,  10.20334),
    'Tram 3':  (45.56305,  10.1966),
    'Tram 4':  (45.56026,  10.19167),
    'Tram 5':  (45.55645,  10.1879),
    'Tram 6':  (45.55224,  10.19236),
    'Tram 7':  (45.54899,  10.19717),
    'Tram 8':  (45.54857,  10.20412),
    'Tram 9':  (45.54683,  10.21176),
    'Tram 10': (45.54472,  10.22047),
    'Tram 11': (45.54497,  10.22472),
    'Tram 12': (45.54094,  10.22326),
    'Tram 13': (45.53667,  10.22232),
    'Tram 14': (45.53228,  10.22086),
    'Tram 15': (45.53203,  10.21506),
    'Tram 16': (45.53461,  10.21136),
    'Tram 17': (45.53126,  10.20817),
    'Tram 18': (45.52559,  10.20349),
    'Tram 19': (45.52667,  10.19817),
    'Tram 20': (45.52795,  10.19355),
    'Tram 21': (45.52965,  10.1878),
    'Tram 22': (45.53088,  10.18325),
    'Tram 23': (45.53021,  10.17677),
}

In [ ]:
# Build arc lists for metro and tram (both directions)
metro_keys = list(metro_stations.keys())
tram_keys  = list(tram_stations.keys())

metro_arcs_back  = [{'start': s1, 'end': s2, 'capacity': 50} for s1, s2 in zip(metro_keys[:-1], metro_keys[1:])]
metro_arcs_forth = [{'start': s2, 'end': s1, 'capacity': 50} for s1, s2 in zip(metro_keys[:-1], metro_keys[1:])]
tram_arcs_back   = [{'start': s1, 'end': s2, 'capacity': 50} for s1, s2 in zip(tram_keys[:-1],  tram_keys[1:])]
tram_arcs_forth  = [{'start': s2, 'end': s1, 'capacity': 50} for s1, s2 in zip(tram_keys[:-1],  tram_keys[1:])]

# Transfer arcs between metro and tram
transfert_arcs = [
    {'start': 'San Faustino', 'end': 'Tram 10',     'capacity': 3000},
    {'start': 'Tram 10',      'end': 'San Faustino', 'capacity': 3000},
    {'start': 'Stazione FS',  'end': 'Tram 15',      'capacity': 3000},
    {'start': 'Tram 15',      'end': 'Stazione FS',  'capacity': 3000},
]

# Combine all arcs
arcs = metro_arcs_back + tram_arcs_back + metro_arcs_forth + tram_arcs_forth + transfert_arcs
print(f"Total arcs: {len(arcs)}")

## 3. Timetable (Peak / Off-Peak)

In [ ]:
peak_start        = time(7, 0)   # 7 AM
peak_end          = time(19, 0)  # 7 PM
peak_frequency    = 10           # minutes during peak
off_peak_frequency = 20          # minutes during off-peak

def get_current_time():
    return datetime.now().time()

def is_peak_time(current_time):
    return peak_start <= current_time <= peak_end

def generate_timetable():
    current_time = get_current_time()
    if is_peak_time(current_time):
        print("Peak time schedule activated.")
        frequency = peak_frequency
    else:
        print("Off-peak schedule activated.")
        frequency = off_peak_frequency
    return list(range(0, 19 * 60, frequency))

T = generate_timetable()
print(f"Number of time slots: {len(T)}")

## 4. Graph & Dijkstra's Algorithm

In [ ]:
class Graph:
    def __init__(self, arcs):
        """
        arcs: List of tuples (origin, destination, weight)
        """
        self.graph = {}
        for origin, destination, weight in arcs:
            if origin not in self.graph:
                self.graph[origin] = []
            if destination not in self.graph:
                self.graph[destination] = []
            self.graph[origin].append((destination, weight))

    def dijkstra(self, start, end):
        priority_queue = [(0, start, [])]
        visited = set()
        shortest_paths = {start: (0, [])}

        while priority_queue:
            (current_distance, current_node, path) = heapq.heappop(priority_queue)
            if current_node in visited:
                continue
            visited.add(current_node)

            if current_node == end:
                return path

            for neighbor, weight in self.graph.get(current_node, []):
                distance_new = current_distance + weight
                if neighbor not in shortest_paths or distance_new < shortest_paths[neighbor][0]:
                    shortest_paths[neighbor] = (distance_new, path + [(current_node, neighbor, weight)])
                    heapq.heappush(priority_queue, (distance_new, neighbor, path + [(current_node, neighbor, weight)]))

        return None  # No path found


def geodesic_distance(point1, point2):
    return geodesic(point1, point2).km


# Build graph from arc list
archi = [(arc['start'], arc['end'], 1) for arc in arcs]
graph = Graph(archi)
print("Graph built successfully.")

## 5. Generate Restaurants & Customers

In [ ]:
def generate_points(center, num_points, sigma):
    return [
        (np.random.normal(center[0], sigma), np.random.normal(center[1], sigma))
        for _ in range(num_points)
    ]

def reduceTo(lista, numb):
    if len(lista) <= numb:
        return lista
    newlist = []
    inserted = 0
    while inserted < numb:
        rand = random.randint(1, len(lista) - 1)
        if lista[rand] not in newlist:
            newlist.append(lista[rand])
            inserted += 1
    return newlist

sigma = 0.01
restaurants = []
customers   = []

for mstation in metro_stations.values():
    restaurants.extend(generate_points(mstation, 5, sigma))
    customers.extend(generate_points(mstation, 10, sigma))

for tstation in tram_stations.values():
    restaurants.extend(generate_points(tstation, 5, sigma))
    customers.extend(generate_points(tstation, 10, sigma))

restaurants = reduceTo(restaurants, 10)
customers   = reduceTo(customers,   10)

print(f"Restaurants: {len(restaurants)}, Customers: {len(customers)}")

## 6. Generate Orders & Arc-Usage Matrix (B)

In [ ]:
def get_key_from_value(d, val):
    keys = [k for k, v in d.items() if v == val]
    return keys[0] if keys else None

# Orders: pairs (restaurant, customer) more than 1.6 km apart
Orders = [(i, j) for i in restaurants for j in customers if geodesic_distance(i, j) > 1.6]
OrdersName = [f'OrderN_{k}' for k, _ in enumerate(Orders)]

print(f"Number of orders: {len(Orders)}")

# Arc-usage binary matrix B[arc, order]
B = np.zeros((len(archi), len(Orders)))

for o, (rest, cust) in enumerate(Orders):
    # Find nearest transit station to restaurant
    minLength = float('inf')
    nodeStart = None
    for mstation in metro_stations.values():
        length = geodesic_distance(rest, mstation)
        if length < minLength:
            nodeStart = get_key_from_value(metro_stations, mstation)
            minLength = length
    for tstation in tram_stations.values():
        length = geodesic_distance(rest, tstation)
        if length < minLength:
            nodeStart = get_key_from_value(tram_stations, tstation)
            minLength = length

    # Find nearest transit station to customer
    minLength = float('inf')
    nodeEnd = None
    for mstation in metro_stations.values():
        length = geodesic_distance(cust, mstation)
        if length < minLength:
            nodeEnd = get_key_from_value(metro_stations, mstation)
            minLength = length
    for tstation in tram_stations.values():
        length = geodesic_distance(cust, tstation)
        if length < minLength:
            nodeEnd = get_key_from_value(tram_stations, tstation)
            minLength = length

    traversed_arcs = graph.dijkstra(nodeStart, nodeEnd)
    if traversed_arcs:
        for travarc in traversed_arcs:
            for arc_idx, arc_tuple in enumerate(archi):
                if arc_tuple == travarc:
                    B[arc_idx][o] = 1

# Demand per order
d_p = [np.random.randint(1, 5) for _ in range(len(Orders))]
print("B matrix and demand vector generated.")

## 7. Network Map Visualization

In [ ]:
# Download OSM street network
G = ox.graph_from_bbox(north, south, east, west, network_type='all')

fig, ax = ox.plot_graph(
    G, bbox=bbox,
    node_color="white", edge_color="lightgray",
    bgcolor="white", show=False, close=False
)

# Metro line
metro_x, metro_y = zip(*metro_stations.values())
ax.plot(metro_y, metro_x, color='red',  linewidth=3, label='Metro Line')
ax.scatter(metro_y, metro_x, color='darkred', s=100, marker='o', edgecolor='black')

# Tram line
tram_x, tram_y = zip(*tram_stations.values())
ax.plot(tram_y, tram_x, color='blue', linewidth=3, label='Tram Line')
ax.scatter(tram_y, tram_x, color='darkblue', s=100, marker='o', edgecolor='black')

# Restaurants & customers
rest_x, rest_y = zip(*restaurants)
ax.scatter(rest_y, rest_x, color='orange', s=30, label='Restaurants', edgecolor='black', alpha=0.7)

cust_x, cust_y = zip(*customers)
ax.scatter(cust_y, cust_x, color='green', s=30, label='Customers', edgecolor='black', alpha=0.7)

# Labels
for station, (x, y) in metro_stations.items():
    ax.text(y, x, station, fontsize=7, color='red',  ha='right', weight='bold')
for station, (x, y) in tram_stations.items():
    ax.text(y, x, station, fontsize=7, color='blue', ha='right', weight='bold')

plt.legend()
plt.title('Brescia Transit Network — Restaurants & Customers')
plt.show()

## 8. Optimization Model (Pyomo / GLPK)

In [ ]:
model = pyo.ConcreteModel()

# Decision variable: y[order, time] = 1 if order is assigned to run at time t
model.y = pyo.Var(OrdersName, T, domain=Binary)

# Objective: maximize weighted order completion (earlier runs get higher weight)
model.obj = pyo.Objective(
    expr=sum(
        (19 * 60 - t) / (19 * 60) * model.y[o, t]
        for o in OrdersName for t in T
    ),
    sense=maximize
)

model.constraints = ConstraintList()

# Capacity constraint: total demand on each arc at each time slot <= 20
for t in T:
    for a in range(len(archi)):
        model.constraints.add(
            sum(B[a, o] * model.y[OrdersName[o], t] * d_p[o]
                for o in range(len(Orders))) <= 20
        )

# Assignment constraint: each order assigned to at most one time slot
for o in OrdersName:
    model.constraints.add(sum(model.y[o, t] for t in T) <= 1)

print("Model built successfully.")

In [ ]:
SOLVER_NAME = 'glpk'
TIME_LIMIT  = 60

solver = SolverFactory(SOLVER_NAME)
solver.options['mipgap'] = 0.01

if   'glpk'    in SOLVER_NAME: solver.options['tmlim']        = TIME_LIMIT
elif 'cplex'   in SOLVER_NAME: solver.options['timelimit']    = TIME_LIMIT
elif 'gurobi'  in SOLVER_NAME: solver.options['TimeLimit']    = TIME_LIMIT
elif 'xpress'  in SOLVER_NAME: solver.options['soltimelimit'] = TIME_LIMIT

results = solver.solve(model, tee=True)
print("\nSolver finished.")

## 9. Results

In [ ]:
print("=== ORDER ASSIGNMENT ===")
for o in OrdersName:
    served = False
    for t in T:
        if pyo.value(model.y[o, t]) == 1:
            print(f"  Order {o} → assigned to run {t} min")
            served = True
    if not served:
        print(f"  Order {o} → NOT assigned to any run")

print("\n=== ARC USAGE ===")
for t in T:
    for a in range(len(archi)):
        occ = sum(B[a, o] * pyo.value(model.y[OrdersName[o], t]) * d_p[o]
                  for o in range(len(Orders)))
        if occ > 0:
            print(f"  Arc {arcs[a]['start']} → {arcs[a]['end']} | run {t} min | usage {occ:.0f}/20")